# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [ ]:
df = pd.read_csv('data/AviationData.csv', encoding='latin-1')

print("Shape:", df.shape)
print("\n--- Data Types & Non-Null Counts ---")
df.info()

print("\n--- Summary Statistics ---")
display(df.describe())

print("\n--- Missing Values ---")
display(df.isna().sum())

print("\n--- Column Names ---")
print(df.columns.tolist())

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

#### Aircraft.Category

`value_counts(dropna=False)` reveals that NaN entries make up a significant portion of the data. Because the column was introduced later in the NTSB database, most older records have NaN — but those older records describe conventional airplanes. Dropping all NaN values here would eliminate a large share of usable data. We therefore **keep rows where `Aircraft.Category` is `'Airplane'` OR NaN**, and drop only the non-airplane categories (helicopter, glider, balloon, etc.).

In [ ]:
# --- Aircraft.Category ---
print("Aircraft.Category value counts:")
print(df['Aircraft.Category'].value_counts(dropna=False))

# Keep airplanes and NaN (NaN likely means airplane for older records)
df = df[df['Aircraft.Category'].isin(['Airplane']) | df['Aircraft.Category'].isna()]
print(f"\nAfter Aircraft.Category filter: {df.shape[0]} rows")

# --- Amateur.Built ---
print("\nAmateur.Built value counts:")
print(df['Amateur.Built'].value_counts(dropna=False))

# Keep only professionally built aircraft
df = df[df['Amateur.Built'] == 'No']
print(f"After Amateur.Built filter: {df.shape[0]} rows")

# --- Event.Date ---
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')
print(f"\nDate range before filter: {df['Event.Date'].min()} to {df['Event.Date'].max()}")

# Keep events from 1983 onwards (40-year active window for a plane retired by 2023)
df = df[df['Event.Date'] >= '1983-01-01']
print(f"After date filter (>=1983): {df.shape[0]} rows")
print(f"Date range after filter: {df['Event.Date'].min()} to {df['Event.Date'].max()}")

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [ ]:
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries',
               'Total.Minor.Injuries', 'Total.Uninjured']

print("Missing values in injury columns:")
print(df[injury_cols].isna().sum())
print()

# Assumption: a missing injury count is treated as zero.
# When an accident report omits a count it most likely means no one fell
# into that category.  This is a standard imputation choice for this dataset.
df[injury_cols] = df[injury_cols].fillna(0)

# Total people on board = sum of all injury categories
df['Total.People.On.Board'] = df[injury_cols].sum(axis=1)

# Serious.Fatal.Fraction = (Fatal + Serious) / Total on board
# Only compute where total > 0 to avoid division by zero
df['Serious.Fatal.Fraction'] = np.where(
    df['Total.People.On.Board'] > 0,
    (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['Total.People.On.Board'],
    np.nan
)

# Drop rows where total people on board is 0 (no useful injury data)
before = df.shape[0]
df = df[df['Total.People.On.Board'] > 0]
print(f"Dropped {before - df.shape[0]} rows with 0 people on board.")
print(f"Remaining rows: {df.shape[0]}")

print("\nSerious.Fatal.Fraction summary:")
display(df['Serious.Fatal.Fraction'].describe())

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [ ]:
print("Aircraft.damage value counts (including NaN):")
print(df['Aircraft.damage'].value_counts(dropna=False))

# Standardize casing
df['Aircraft.damage'] = df['Aircraft.damage'].str.strip().str.title()

# Replace unknown / rare values with NaN
df['Aircraft.damage'] = df['Aircraft.damage'].replace({'Unknown': np.nan})

print("\nAfter cleaning:")
print(df['Aircraft.damage'].value_counts(dropna=False))

# Binary column: 1 if Destroyed, 0 otherwise (NaN stays NaN)
df['Destroyed'] = np.where(
    df['Aircraft.damage'].isna(), np.nan,
    np.where(df['Aircraft.damage'] == 'Destroyed', 1, 0)
)

print("\nDestroyed column distribution:")
print(df['Destroyed'].value_counts(dropna=False))

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

**Observed problems with the `Make` column:**

1. **Inconsistent casing** — e.g. `CESSNA`, `cessna`, `Cessna` all refer to the same manufacturer.
2. **Leading/trailing whitespace** — some entries have extra spaces.
3. **Spelling variants of the same manufacturer** — e.g. `Mcdonnell Douglas`, `Mcdonnell-Douglas`, `MCDONNELL DOUGLAS`; `De Havilland`, `Dehavilland`; `Boeing` vs `BOEING`.
4. **Abbreviations and legacy names** — some entries use a division name or abbreviation instead of the parent company.

**Cleaning plan:** strip whitespace → convert to title case → merge known variants with `replace()` → keep only makes with ≥ 50 accidents.

In [ ]:
# Basic cleaning: strip whitespace, title case
df['Make'] = df['Make'].str.strip().str.title()

# Merge common variants into canonical names
make_replacements = {
    'Mcdonnell Douglas': 'Mcdonnell Douglas',
    'Mcdonnell-Douglas': 'Mcdonnell Douglas',
    'Dehavilland': 'De Havilland',
    'De Havilland Canada': 'De Havilland',
    'Bombardier Inc': 'Bombardier',
    'Embraer-Empresa Brasileira De Aeronautica S.A.': 'Embraer',
    'Embraer S.A.': 'Embraer',
    'Empresa Brasileira De Aeronautica S.A. (Embraer)': 'Embraer',
    'Airbus Industrie': 'Airbus',
    'Raytheon Aircraft Company': 'Raytheon',
    'Raytheon Corporate Jets': 'Raytheon',
    'Cirrus Design': 'Cirrus',
    'Cirrus Design Corp': 'Cirrus',
    'Maule Air': 'Maule',
    'Maule Air Inc': 'Maule',
}
df['Make'] = df['Make'].replace(make_replacements)

# Remove helicopter manufacturers that may have slipped in via NaN Aircraft.Category
helicopter_makes = ['Bell', 'Robinson', 'Hughes', 'Schweizer', 'Sikorsky', 'Eurocopter', 'Agusta']
before = df.shape[0]
df = df[~df['Make'].isin(helicopter_makes)]
print(f"Removed {before - df.shape[0]} rows from helicopter-related makes: {helicopter_makes}")

print("\nTop 30 makes after cleaning:")
display(df['Make'].value_counts().head(30))

# Keep only makes with at least 50 accidents
make_counts = df['Make'].value_counts()
valid_makes = make_counts[make_counts >= 50].index
before = df.shape[0]
df = df[df['Make'].isin(valid_makes)]
print(f"\nKept {len(valid_makes)} makes with ≥50 accidents.")
print(f"Rows dropped: {before - df.shape[0]}, remaining: {df.shape[0]}")

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [ ]:
# Drop rows with no model info
print(f"Model NaN count: {df['Model'].isna().sum()}")
df = df.dropna(subset=['Model'])

# Clean model names: strip whitespace, uppercase to merge casing variants
df['Model'] = df['Model'].str.strip().str.upper()

# Check if model names are unique to each make
model_make_counts = df.groupby('Model')['Make'].nunique()
shared_models = model_make_counts[model_make_counts > 1]
print(f"\nModels shared across multiple makes: {len(shared_models)}")
display(shared_models.head(10))

# Since some model names appear under multiple makes,
# create a unique identifier combining make and model
df['Make_Model'] = df['Make'] + ' ' + df['Model']

print(f"\nUnique Make_Model combinations: {df['Make_Model'].nunique()}")
print(f"Remaining rows: {df.shape[0]}")

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [ ]:
other_cols = ['Engine.Type', 'Weather.Condition', 'Number.of.Engines',
              'Purpose.of.flight', 'Broad.phase.of.flight']

for col in other_cols:
    print(f"\n{'='*50}")
    print(f"Column: {col}")
    print(df[col].value_counts(dropna=False))

# --- Engine.Type ---
df['Engine.Type'] = df['Engine.Type'].str.strip().str.title()
df['Engine.Type'] = df['Engine.Type'].replace({
    'Unknown': np.nan, 'Unk': np.nan
})
# Group rare engine types (fewer than 50 occurrences) into "Other"
engine_counts = df['Engine.Type'].value_counts()
rare_engines = engine_counts[engine_counts < 50].index
df['Engine.Type'] = df['Engine.Type'].replace({e: 'Other' for e in rare_engines})

# --- Weather.Condition ---
df['Weather.Condition'] = df['Weather.Condition'].str.strip().str.upper()
df['Weather.Condition'] = df['Weather.Condition'].replace({
    'UNK': np.nan, 'UNKNOWN': np.nan
})

# --- Number.of.Engines ---
print(f"\n\nNumber.of.Engines unique values before cleaning: {sorted(df['Number.of.Engines'].dropna().unique())}")
# 0 engines on an airplane is almost certainly a placeholder → treat as NaN
df['Number.of.Engines'] = df['Number.of.Engines'].replace({0: np.nan})
print(f"Number.of.Engines unique values after cleaning: {sorted(df['Number.of.Engines'].dropna().unique())}")

# --- Purpose.of.flight ---
df['Purpose.of.flight'] = df['Purpose.of.flight'].str.strip().str.title()
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace({
    'Unknown': np.nan
})
# Group rare flight purposes (fewer than 50 occurrences) into "Other"
purpose_counts = df['Purpose.of.flight'].value_counts()
rare_purposes = purpose_counts[purpose_counts < 50].index
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace({p: 'Other' for p in rare_purposes})

# --- Broad.phase.of.flight ---
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].str.strip().str.title()
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].replace({
    'Unknown': np.nan, 'Other': np.nan
})

print("\n\n--- Cleaned value counts ---")
for col in other_cols:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False).head(15))

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [ ]:
# Keep only columns with more than 20,000 non-null values
non_null_counts = df.notna().sum()
print("Non-null counts per column:")
print(non_null_counts.sort_values())

threshold = 20000
cols_to_keep = non_null_counts[non_null_counts > threshold].index.tolist()
cols_to_drop = [c for c in df.columns if c not in cols_to_keep]

print(f"\nColumns dropped (≤{threshold} non-null): {cols_to_drop}")
print(f"Columns kept: {len(cols_to_keep)}")

df = df[cols_to_keep]
print(f"\nFinal shape: {df.shape}")
df.info()

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [ ]:
df.to_csv('data/AviationData_cleaned.csv', index=False)
print("Cleaned data saved to data/AviationData_cleaned.csv")
print(f"Final dataset: {df.shape[0]} rows, {df.shape[1]} columns")